# PCA — do the features separate by resistance mechanism?

Merges the Pfeature and ProFET feature tables on `family` + `sequence_id`, attaches RGI resistance-mechanism labels, and runs PCA to check whether mechanism classes separate before committing to model building.

**Read this before interpreting the plot:**

1. **Redundancy will fake structure.** If the input still contains near-identical sequences, PCA clusters will largely reflect *sequence similarity*, not mechanism — a family with 40 near-duplicate variants becomes a tight blob that looks like a real cluster and isn't. Cell 4 subsets to CD-HIT representatives if you supply a cluster mapping, and warns loudly if you don't.
2. **Expect PC1 to be hydrophobicity, not mechanism.** Composition and biophysical features mostly encode membrane-vs-soluble character. Efflux pumps are polytopic membrane proteins; beta-lactamases are soluble periplasmic enzymes. Separation along that axis is real but is a *localisation* signal, not evidence that these features resolve mechanism per se. Cell 8 prints the top PC loadings so you can check which it is rather than guess.
3. **PCA is linear and unsupervised.** Poor separation here doesn't mean a supervised non-linear model will fail — it means there's no dominant linear axis. Treat this as a sanity check, not a go/no-go.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

pd.set_option("display.width", 200)
print(f"pandas {pd.__version__}")

In [ ]:
# --- inputs ---
PFEATURE_TABLE = Path("results/pfeature_features/all_families_features.csv")
PROFET_TABLE   = Path("results/profet_features/all_families_profet_features.csv")

# RGI output (or any table mapping sequence -> mechanism)
LABEL_TABLE    = Path("rgi_labels.csv")
LABEL_ID_COL   = "ORF_ID"                  # column in LABEL_TABLE matching sequence_id
LABEL_COL      = "Resistance Mechanism"    # column holding the mechanism

# Optional CD-HIT representative list: a text file with one representative sequence_id
# per line. Set to None to skip (and get a warning instead).
CDHIT_REPS     = None

OUTPUT_DIR = Path("results/pca")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KEY = ["family", "sequence_id"]

In [ ]:
pf = pd.read_csv(PFEATURE_TABLE)
pr = pd.read_csv(PROFET_TABLE)
print(f"Pfeature: {pf.shape[0]} rows x {pf.shape[1]-len(KEY)} features")
print(f"ProFET:   {pr.shape[0]} rows x {pr.shape[1]-len(KEY)} features")

# inner merge, but report what each side loses rather than silently dropping
merged = pf.merge(pr, on=KEY, how="inner", validate="one_to_one")
lost_pf = len(pf) - len(merged)
lost_pr = len(pr) - len(merged)
print(f"\nMerged: {merged.shape[0]} rows x {merged.shape[1]-len(KEY)} features")
if lost_pf or lost_pr:
    print(f"  WARNING: {lost_pf} Pfeature row(s) and {lost_pr} ProFET row(s) had no counterpart and were dropped.")
    print("  If this is non-zero, the two tables were built from different FASTA sets -- check before trusting the PCA.")
else:
    print("  Both tables aligned exactly, no rows dropped.")

In [ ]:
labels = pd.read_csv(LABEL_TABLE)[[LABEL_ID_COL, LABEL_COL]].rename(
    columns={LABEL_ID_COL: "sequence_id", LABEL_COL: "mechanism"}
)
labels = labels.drop_duplicates(subset="sequence_id")

data = merged.merge(labels, on="sequence_id", how="left")
missing = data["mechanism"].isna().sum()
print(f"{len(data) - missing}/{len(data)} sequences have a mechanism label.")
if missing:
    print(f"  {missing} unlabelled sequence(s) -- excluded from PCA (can't colour or score them).")
    data = data[data["mechanism"].notna()].reset_index(drop=True)

print("\nClass balance:")
counts = data["mechanism"].value_counts()
for m, c in counts.items():
    print(f"  {m}: {c}")
if (counts < 10).any():
    print("\n  NOTE: classes under ~10 members will look like clusters purely from sparsity. Interpret those with care.")

In [ ]:
# Optional deduplication to CD-HIT representatives
if CDHIT_REPS is not None:
    reps = {ln.strip() for ln in open(CDHIT_REPS) if ln.strip()}
    before = len(data)
    data = data[data["sequence_id"].isin(reps)].reset_index(drop=True)
    print(f"Subset to CD-HIT representatives: {before} -> {len(data)} sequences")
else:
    print("!! No CD-HIT representative list supplied (CDHIT_REPS = None).")
    print("!! PCA will run on ALL sequences, including near-duplicates.")
    print("!! Tight clusters in the plot below may reflect sequence redundancy rather than mechanism.")

In [ ]:
meta_cols = KEY + ["mechanism"]
X_df = data.drop(columns=meta_cols).select_dtypes(include=[np.number])
print(f"Starting matrix: {X_df.shape[0]} x {X_df.shape[1]}")

# 1. columns that are entirely NaN -- unrecoverable, drop
all_nan = X_df.columns[X_df.isna().all()].tolist()
if all_nan:
    X_df = X_df.drop(columns=all_nan)
    print(f"  dropped {len(all_nan)} all-NaN column(s)")

# 2. remaining NaNs -- median-impute, and report scale so you know what was fabricated
n_nan = int(X_df.isna().sum().sum())
if n_nan:
    affected = X_df.columns[X_df.isna().any()].tolist()
    print(f"  imputing {n_nan} NaN cell(s) with column medians, across {len(affected)} column(s):")
    for c in affected:
        print(f"      {c}: {int(X_df[c].isna().sum())} missing")
    X_df = X_df.fillna(X_df.median(numeric_only=True))

# 3. zero-variance columns -- carry no information and break StandardScaler (divide by zero)
zero_var = X_df.columns[X_df.std(numeric_only=True) == 0].tolist()
if zero_var:
    X_df = X_df.drop(columns=zero_var)
    print(f"  dropped {len(zero_var)} zero-variance column(s)")

# 4. standardise -- mandatory here: molecular_weight_da is ~1e4, aromaticity is ~1e-2.
#    Without this, PCA would be driven almost entirely by whichever features have the largest raw units.
X = StandardScaler().fit_transform(X_df.values)
y = data["mechanism"].values
print(f"\nFinal matrix for PCA: {X.shape[0]} sequences x {X.shape[1]} features")

In [ ]:
n_comp = min(20, X.shape[0], X.shape[1])
pca_full = PCA(n_components=n_comp).fit(X)
evr = pca_full.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, len(evr)+1), evr*100)
axes[0].set_xlabel("Principal component"); axes[0].set_ylabel("% variance explained")
axes[0].set_title("Scree")
axes[1].plot(range(1, len(evr)+1), np.cumsum(evr)*100, marker="o")
axes[1].axhline(80, ls="--", c="grey", lw=1)
axes[1].set_xlabel("Components"); axes[1].set_ylabel("Cumulative % variance")
axes[1].set_title("Cumulative variance (dashed = 80%)")
plt.tight_layout(); plt.show()

print(f"PC1: {evr[0]*100:.1f}%   PC2: {evr[1]*100:.1f}%   PC1+PC2: {(evr[0]+evr[1])*100:.1f}%")
n80 = int(np.searchsorted(np.cumsum(evr), 0.80) + 1)
print(f"Components needed for 80% variance: {n80}")
if (evr[0]+evr[1]) < 0.30:
    print("\nNOTE: PC1+PC2 hold less than 30% of the variance -- a 2D scatter is a poor summary here.")
    print("Weak visual separation below would NOT be strong evidence that the classes are inseparable.")

In [ ]:
scores = pca_full.transform(X)

fig, ax = plt.subplots(figsize=(8, 6.5))
for mech in sorted(pd.unique(y)):
    m = y == mech
    ax.scatter(scores[m, 0], scores[m, 1], label=f"{mech} (n={m.sum()})", alpha=0.7, s=40)
ax.set_xlabel(f"PC1 ({evr[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({evr[1]*100:.1f}%)")
ax.set_title("PCA — coloured by resistance mechanism")
ax.legend(fontsize=8, loc="best")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pca_by_mechanism.png", dpi=150)
plt.show()

pd.DataFrame({
    "family": data["family"], "sequence_id": data["sequence_id"], "mechanism": y,
    "PC1": scores[:, 0], "PC2": scores[:, 1], "PC3": scores[:, 2] if scores.shape[1] > 2 else np.nan,
}).to_csv(OUTPUT_DIR / "pca_scores.csv", index=False)
print(f"Scores -> {OUTPUT_DIR/'pca_scores.csv'}")

In [ ]:
# Silhouette: how well-separated are the mechanism classes, quantitatively?
# Range -1 to 1. ~0 means classes overlap heavily; >0.25 is meaningful structure.
# Computed on the full scaled matrix, not just PC1/PC2 -- the 2D view can hide separation.
if len(np.unique(y)) > 1:
    sil_full = silhouette_score(X, y)
    sil_2d = silhouette_score(scores[:, :2], y)
    print(f"Silhouette (all {X.shape[1]} scaled features): {sil_full:.3f}")
    print(f"Silhouette (PC1+PC2 only):                    {sil_2d:.3f}")
    print()
    if sil_full < 0.10:
        print("Near zero -> mechanism classes overlap heavily in this feature space.")
        print("A linear model on these features alone is unlikely to separate them well.")
    elif sil_full < 0.25:
        print("Weak structure -> some signal, heavily overlapping. Non-linear models may still do fine.")
    else:
        print("Clear structure -> classes occupy distinguishable regions.")
    print("\nCheck the loadings below before concluding this reflects MECHANISM rather than")
    print("membrane-vs-soluble character, which these features encode strongly.")
else:
    print("Only one mechanism class present -- silhouette undefined.")

In [ ]:
# What actually drives PC1 and PC2? This is the check on caveat #2 in the header.
loadings = pd.DataFrame(
    pca_full.components_[:2].T, index=X_df.columns, columns=["PC1", "PC2"]
)
for pc in ["PC1", "PC2"]:
    print(f"\nTop 15 features loading on {pc}:")
    top = loadings[pc].abs().sort_values(ascending=False).head(15)
    for feat in top.index:
        print(f"   {loadings.loc[feat, pc]:+.3f}  {feat}")

loadings.to_csv(OUTPUT_DIR / "pca_loadings.csv")
print(f"\nFull loadings -> {OUTPUT_DIR/'pca_loadings.csv'}")
print("\nIf gravy / N_gravy / aliphatic_index / hydrophobic AAC columns dominate PC1,")
print("that axis is hydrophobicity -- i.e. membrane vs soluble, not mechanism.")